# Module 04 — Finding Failure Modes

> LangSmith UI focused — no code cells in this notebook.

You have traces from Module 1, evaluation scores from Modules 2–3, and (after Module 5) annotation data.
This module covers how to turn that raw trace data into signal that drives agent improvements — without
manually reading every trace.


## The shift in scale

Manually reviewing traces worked when agents were short and few. Three trends pushed past that:

- **Agents are longer-running** — a single trace spans many tool calls and subagent invocations.
- **Inputs/outputs are multimodal** — images, attachments, documents, not just text.
- **Agents are proliferating** — many agents, each with its own trace stream.

LangSmith ships three features for this, each turning a different slice of trace data into action:

- **Chat** — a workspace AI assistant; ask natural-language questions grounded in your traces, threads,
  prompts, and experiments.
- **Insights Agent** — an automated job that categorizes a trace corpus and surfaces patterns nobody knew
  to look for.
- **Engine** — a closed-loop system that detects recurring failures, diagnoses root causes, proposes fixes
  as pull requests, red-teams the agent, and verifies fixes. Available on **cloud and self-hosted**, using
  LangChain-managed inference **or your own model (bring-your-own-model)**.


---

## Engine

> Engine is enabled per org (**Settings → Engine**). It runs on cloud **and self-hosted**. If you don't
> see it in your project, ask your LangSmith admin to enable it.


### What Engine does

Engine plugs into the full lifecycle: trace → recurring failure detected → root cause diagnosed → fix
proposed → evaluator deployed → dataset examples generated. It scans your project's traces on a schedule
(every 6 hours by default) and surfaces issues without manual review.

For each issue it links the relevant traces, proposes a fix, generates a custom evaluator to catch
regressions, and creates ground-truth dataset examples from the production inputs. Where **Insights** gives
a categorized view of *what's happening*, **Engine** surfaces *what needs to change*.

Engine can run on **LangChain-managed inference or your own model** — configure bring-your-own-model in
Engine settings if you'd rather keep inference on your own provider/keys.


### Enabling Engine

Two steps — org-level then per-project:

1. **Org admin** enables Engine in **Settings → Engine → Engine enablement**, and chooses the inference
   mode (LangChain-managed **or** your own model / BYOM).
2. **Per project**, open the **Engine tab** and configure:
   - Optionally connect a **GitHub repository** (recommended) — Engine uses your source to diagnose more
     accurately and open PRs directly from issues.
   - Select **priority categories** — for the Northwind Mobile concierge: **Hallucination**, **PII
     Leakage**, **Tool Call Failures**. Add custom concerns with **+ Add something specific**.
   - Click **Start Analyzing** (first run can take ~20 min).
3. **Review the agent overview document** Engine generates from your traces — it's the context for all
   analysis, so accuracy here drives issue quality. Edit it anytime from Engine settings.

<img src="../images/engine_config.png" alt="engine_config.png" style="width: auto; max-height: 380px; border-radius: 8px;">


### Reviewing an issue

Click an issue to open its detail panel. A **diagnosis** describes the problem and impact, backed by
linked traces. Four sections:

- **Linked traces** — the traces supporting the diagnosis (e.g. the concierge reading back an SSN, or
  inventing a plan price).
- **Proposed Fix** — code/prompt changes targeting the root cause (higher quality with a repo connected).
- **Suggested Evaluator** — a ready-to-deploy run rule to catch this failure mode going forward. If it
  fires after you close the issue, Engine reopens it.
- **Offline Examples** — dataset examples from the production inputs that triggered the issue, each with
  named **assertions** about what a correct response should/shouldn't include.

<img src="../images/engine_issue.png" alt="engine_issue.png" style="display: block; width: 67%; max-height: 400px; border-radius: 8px; margin-bottom: 1rem;">


### Taking action

From the issue panel:

- **Open PR** — opens a GitHub PR with the fix applied (Deep Agents / LangChain / LangGraph repos).
- **Create Evaluator** — deploys the suggested evaluator as an online evaluator (see Module 2). Toggle
  **Apply to past runs** to preview how many historical traces it would flag.
- **Add offline examples** — adds examples to a dataset (see Module 3), optionally via an annotation queue
  (Module 5) to review the assertions first.
- **Copy Fix Context** — copies the full issue as a prompt for an external LLM / coding assistant.
- **Resolve / Ignore** — with a reason that feeds back into Engine. Resolved issues auto-reopen if they
  resurface.

<img src="../images/engine_triage.png" alt="engine_triage.png" style="display: block; width: auto; max-height: 420px; border-radius: 8px; margin-bottom: 1rem;">


### Red teaming

Beyond analyzing production traces, Engine can **proactively probe** the agent for failures before users
hit them. On the Engine tab, open **Red Teaming** and click **Run red team**. Engine generates adversarial
prompt pairs, runs them against your agent, and reports **product-facing findings** — correctness slips,
guardrail gaps, policy/positioning issues — each with:

- the **probe** (Prompt 1 / Prompt 2) and whether it **worked** or was **blocked**,
- **Observed** vs **Expected** behavior, a severity, and a *Differential Behavior* tag,
- an **Unexpected Finding** flag when a probe surfaces something the judge didn't anticipate.

Add any finding **to a dataset** to lock it in as a regression test.

<img src="../images/engine_redteam.png" alt="engine_redteam.png" style="display: block; width: auto; max-height: 420px; border-radius: 8px; margin-bottom: 1rem;">


### Testing a fix against your deployment

When your agent is **deployed** (Module 1's `langgraph dev`, or a LangSmith Deployment) and a fix PR is
open, Engine **verifies the fix actually works**. It replays the flagged traces through both the
**baseline** and the **fixed** version and reports per-trace results:

- **Baseline** — did the issue reproduce? (*Issue reproduced* / *Couldn't reproduce*)
- **After PR #…** — did the fix resolve it? (*Fix worked*)

…with a pass count (e.g. **5/5**) and a **Fix worked** summary. **View Experiment** opens the full
comparison. This closes the loop: detect → propose fix → **prove it works on real traces** before merging.

<img src="../images/engine_testfix.png" alt="engine_testfix.png" style="display: block; width: auto; max-height: 360px; border-radius: 8px; margin-bottom: 1rem;">


---

## Chat

> Enabled per org by your LangSmith admin.

Chat is an AI assistant embedded in the LangSmith workspace (bottom-right of every observability, prompt,
and evaluation page). It answers natural-language questions about whatever you're looking at — traces,
threads, prompts, experiments, datasets, queues — using the same data the UI sees. The value is skipping
the filter expression / scrolling / opening-many-traces steps.

<img src="../images/chat.png" alt="chat.png" style="width: auto; max-height: 380px; border-radius: 8px;">


### Accessing Chat

- **`Cmd+I`** (Mac) / **`Ctrl+I`** (Win/Linux) — toggle the panel from any page.
- **`Cmd+Shift+O` / `Ctrl+Shift+O`** — start a fresh conversation.
- Or click the Chat icon in the bottom-right corner.


### Questions worth asking against the telecom project

Open the relevant page first, then ask:

- **On a single trace:** *"Summarize the tool calls the agent made in this trace."*
- **On the project (after Module 2 evaluators ran):** *"Which runs failed the hallucination or PII eval?
  Summarize why."*
- **On the project:** *"Did any customers get an account-not-found response?"* — surfaces the `ACCT-9999`
  lookups.
- **On the project:** *"What's the latency distribution for runs that delegated to the network-diagnostics
  subagent?"*
- **On a dataset's Experiments tab (Module 3):** *"Compare this experiment to the previous one. What
  changed?"*

Chat can also edit prompts conversationally in the Prompt/Context Hub (e.g. *"make the hallucination judge
stricter on sourcing figures"*).


---

## Insights Agent

> Enabled per org by your LangSmith admin.

Insights runs against a project's trace corpus and groups runs into a hierarchy of categories — top-level
themes (e.g. *account lookups*, *outage/troubleshooting*, *billing questions*, *unknown accounts*) with
nested subcategories — plus an executive summary, distribution bars, and per-category aggregates (latency,
error rate, cost, evaluator feedback). Use **Insights** to find what's worth investigating; use **Chat** to
dig into specific traces once you know where to look.


### Prerequisites

**A healthy trace corpus.** The seeding in Module 1, the experiments in Module 3, and the trigger prompts
in Modules 2 and 5 produce enough volume for Insights to find patterns. Production traffic accrues traces
continuously and needs no manual seed.


### Configuring a report

1. Open **Tracing Projects** and select your project (`telecom-concierge-<you>`).
2. **+ New → New Insights Report**.
3. Use the defaults unless you want a specific configuration — it auto-generates a category structure.
4. Launch. The job runs in the background (typically < 30 min).

<img src="../images/insights_config.png" alt="insights_config.png" style="width: auto; max-height: 380px; border-radius: 8px;">


### Reading the report

Opens to an executive summary — the largest patterns by run share. From there:

- **Click a category** to drill into subcategories and sample traces.
- **Distribution bars** show run counts per category.
- **Per-category aggregates** show error rate, latency, cost, and evaluator feedback. High frequency *and*
  high error rate = first place to look.
- **Sample traces** link straight to the trace tree.

<img src="../images/insights_report.png" alt="insights_report.png" style="width: auto; max-height: 420px; border-radius: 8px;">


### Scheduling & acting on findings

Reports can re-run daily / weekly / on a cron. When a category looks actionable:

- **High-frequency failure category** → promote sample traces to an annotation queue (Module 5) for
  labeling; the labels become eval data.
- **Latency-outlier category** → open a sample trace and use **Chat** to summarize the tree, then tune the
  offending tool/prompt.
- **Hallucination / PII pattern** → tighten the judge (Module 2) and add examples to a dataset (Module 3).


## Recap

| Tool | When to use | Self-hosted |
|---|---|---|
| **Engine** | Closed-loop detection, fix proposal, red teaming, fix verification, dataset generation. | **Yes** (cloud + self-hosted; LangChain-managed or BYOM). |
| **Chat** | Ad-hoc questions during development or review. Lowest setup. | Confirm with workspace admin. |
| **Insights Agent** | Automated periodic discovery of patterns across a trace corpus. | Yes. |

Next: `05_annotation_queues.ipynb` — route low-scoring runs to human review.
